#### Recommender

##### Importing Libraries & Loading Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
project_root = Path.cwd().parent
processed_dir = project_root/'data'/'processed'

In [3]:
features = pd.read_parquet(processed_dir/'station_final.parquet')
features_scaled = pd.read_parquet(processed_dir/'station_features_scaled.parquet')

##### Cosine Similarity

In [4]:
similarity = cosine_similarity(features_scaled.values)

In [5]:
similarity_df = pd.DataFrame(
    similarity,
    index=features_scaled.index,
    columns=features_scaled.index
)

In [6]:
similarity_df.head()

nlc,750,1404,3000,500,502,503,850,505,506,5397,...,6954,769,7467,771,573,6560,5131,6561,6562,9846
nlc,,,,,,,,,,,,,,,,,,,,,
750,1.000000,0.767775,-0.486997,-0.006924,0.122979,0.195037,0.844111,0.327226,-0.476507,-0.112217,...,-0.470508,-0.261659,0.535189,-0.688913,-0.034844,0.008539,-0.228942,0.136412,-0.700008,0.486196
1404,0.767775,1.000000,-0.235374,-0.016086,0.194291,-0.037427,0.405680,0.315822,0.052452,0.167457,...,0.025637,-0.095847,0.494082,-0.300836,-0.161524,-0.071640,-0.122827,-0.292003,-0.715741,0.073238
3000,-0.486997,-0.235374,1.000000,0.005063,-0.503529,-0.351542,-0.622853,0.051387,0.661098,0.776284,...,0.777704,0.305154,-0.055883,0.826625,-0.050808,-0.574954,0.170039,-0.211664,0.420230,-0.833911
500,-0.006924,-0.016086,0.005063,1.000000,0.212104,0.844590,-0.000425,-0.007531,-0.010052,-0.008040,...,-0.007954,0.909324,0.057152,-0.000301,0.756398,0.006251,0.940580,0.019768,0.006004,0.003170
502,0.122979,0.194291,-0.503529,0.212104,1.000000,0.520738,0.012963,-0.451448,-0.103102,-0.446082,...,-0.329935,0.018871,-0.304581,-0.347281,0.023587,0.854753,0.120337,0.038004,-0.311725,0.184858


In [8]:
similarity_df.shape

(432, 432)

In [9]:
names = features['station_name']

In [11]:
# Name Lookup, We can use any station we like in query. For example, Brixton.

query = 'Brixton'
matches = names[names.str.contains(query)]
print(matches)

nlc
778    Brixton LU
Name: station_name, dtype: str


In [20]:
# Turn query name into NLC ( National Location Code)
match_nlc = names[names.str.contains(query)].index[0]

# Read the chosen station's similarity row, drop itself and take the top 5 most similar stations
top_sim = similarity_df[match_nlc].drop(match_nlc).sort_values(ascending=False).head(5)

print(f'Stations most similar to {names[match_nlc]}:\n')
for other_nlc, score in top_sim.items():
    print(f"  {names[other_nlc]:28s}  {features.loc[other_nlc, 'archetype']:28s}  similarity={score:.2f}")

Stations most similar to Brixton LU:

  Clapham Common                Inner Local Stations          similarity=0.82
  Tooting Broadway              Inner Local Stations          similarity=0.78
  Manor House                   Inner Local Stations          similarity=0.73
  Dalston Junction              Inner Local Stations          similarity=0.72
  Clapham North                 Inner Local Stations          similarity=0.71


In [21]:
# Saving
similarity_df.to_parquet(processed_dir/'station_similarity.parquet')